In [1]:
!pip install ydata_profiling matplotlib

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.1/400.1 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 687.8/687.8 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.9 MB/s eta 0:00:00
  Created wheel for htmlmin: filename=htmlmin-0.1.12-py3-none-any.whl size=27081 sha256=7ecac81b2b350d347b5d5b726f010ae8fa622cafae0c2ea295a2173e6bf7ff8c
  Stored in directory: /root/.cache/pip/wheels/8d/55/1a/19cd535375ed1ede0c996405ebffe34b196d78e2d9545723a2
Successfully built htmlmin


In [2]:
import pandas as pd
from ydata_profiling import ProfileReport

In [3]:
dados = pd.read_csv("https://raw.githubusercontent.com/MachineTeachingEdu/JAI2025-IA-Educacao/refs/heads/main/An%C3%A1lise%20de%20Dados/dataset.txt", sep='\t')

In [4]:
def limpeza_inicial(dados):
  # prompt: drop all df columns that are all NaN
  dados.dropna(axis=1, how='all', inplace=True)

  # prompt: drop all df columns that have a single unique value
  columns_to_drop = []
  for col in dados.columns:
      if dados[col].nunique() == 1:
          columns_to_drop.append(col)
  dados.drop(columns=columns_to_drop, inplace=True)

  dados.drop(["Row", "Transaction Id", "Tutor Response Type"], axis=1, inplace=True)

  # prompt: transform a datetime column called "Problem Start Time" in a pandas df into seconds from 1970
  dados["Problem Start Time"] = pd.to_datetime(dados["Problem Start Time"]).astype('int64') // 10**9
  return dados

dados = limpeza_inicial(dados)

In [5]:
profile = ProfileReport(dados,title="Computação")
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 19/19 [00:01<00:00, 10.22it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
habilidade_aluno = dados[['KC (Intro CS Concepts)', "Anon Student Id"]]
habilidade_aluno = habilidade_aluno.drop_duplicates()
print("Quantidade de habilidades no dataset: ", len(habilidade_aluno['KC (Intro CS Concepts)'].unique()))
habilidades = habilidade_aluno['KC (Intro CS Concepts)'].unique()
habilidade_aluno['KC (Intro CS Concepts)'].value_counts()

Quantidade de habilidades no dataset:  10


,count
KC (Intro CS Concepts),
Class and Functions,89
Data Types and Structures,87
Variables,78
Programming Language Fundamentals,78
Arithmetic Operations,76
Comparison,50
Boolean Operation,32
Boolean Operations,32
Loops,32


In [7]:
def transforma_dataset(aluno_id, habilidades):
  aluno = dados[(dados['Anon Student Id'] == aluno_id) & (dados['Is Last Attempt'] == 1)]
  habilidade_aluno = aluno['KC (Intro CS Concepts)'].unique()
  dataset_aluno = {"Id": aluno_id}
  # print(f"{aluno_id}")
  for habilidade in habilidade_aluno:
    tem_habilidade = aluno[aluno['KC (Intro CS Concepts)'] == habilidade]
    if len(tem_habilidade) > 0:
      sucesso = tem_habilidade[tem_habilidade['Outcome'] == 'CORRECT'].count()['Outcome']
      total = tem_habilidade.count()['Outcome']
      # print(f"{sucesso} / {total} = {sucesso/total}")
      dataset_aluno[habilidade] = sucesso/total
    else:
      dataset_aluno[habilidade] = 0.1
  return pd.DataFrame.from_dict([dataset_aluno], orient='columns')

dados_final = pd.DataFrame()
for aluno_id in dados['Anon Student Id'].unique():
  dados_aluno = transforma_dataset(aluno_id, habilidades)
  dados_final = pd.concat([dados_final, dados_aluno])
dados_final = dados_final.fillna(0)

In [8]:
dados_final.head()

,Id,Data Types and Structures,Arithmetic Operations,Programming Language Fundamentals,Class and Functions,Variables,Comparison,Boolean Operation,Boolean Operations,Loops,Conditions
0,S002,0.833333,0.857143,0.800000,0.888889,0.857143,0.666667,1.0,1.00,0.0,0.0
0,S011,0.846154,0.765957,0.903226,0.902439,0.758621,0.812500,1.0,0.75,0.5,1.0
0,S014,0.923077,0.750000,0.937500,0.961538,0.923077,1.000000,0.0,0.00,0.0,0.0
0,S020,0.833333,0.833333,0.800000,0.967742,0.789474,0.666667,0.0,0.00,0.0,0.0
0,S023,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.0,0.50,0.0,0.0


In [9]:
dados_final.to_csv('dados_final_habilidades.csv', index=False)

In [10]:
profile = ProfileReport(dados_final,title="Dataset Transformado")
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 11/11 [00:00<00:00, 106.61it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]